# Call the deployed kiosk agent

This notebook calls Amazon Bedrock AgentCore in Stockholm using the local AWS profile `kiosk`. No local FastAPI server is needed. AWS credentials stay in your AWS profile; the OpenRouter key stays in the runtime configuration.

Run the cells in order. Chat calls incur AWS and model-provider charges. Cloud orders currently use temporary SQLite storage.


In [ ]:
%pip install --quiet --upgrade boto3


## Connect with your AWS profile

This is the same profile verified using `aws sts get-caller-identity --profile kiosk`. The endpoint ARN is split into a runtime ARN and the endpoint name `DEFAULT`.


In [ ]:
import json
from uuid import uuid4

import boto3
from botocore.exceptions import ClientError

PROFILE = "kiosk"
REGION = "eu-north-1"
RUNTIME_ARN = (
    "arn:aws:bedrock-agentcore:eu-north-1:463277661277:"
    "runtime/kiosk_agent_core-ZyqeDtANrR"
)
ENDPOINT = "DEFAULT"

aws_session = boto3.Session(profile_name=PROFILE, region_name=REGION)
identity = aws_session.client("sts").get_caller_identity()
print("Account:", identity["Account"])
print("Caller:", identity["Arn"])
assert identity["Account"] == "463277661277", "Check the AWS profile/account."

client = aws_session.client("bedrock-agentcore")


## Start a conversation

Run this cell once per conversation. Re-running it creates a new session with a separate draft and history. A UUID meets AgentCore's session ID length requirement.


In [ ]:
session_id = str(uuid4())
print("Session:", session_id)


## Define the API call

`invoke_agent_runtime` authenticates the request with AWS credentials and sends the JSON payload to our backend's `/invocations` handler. Reusing the session ID preserves conversation context while the runtime session remains available.


In [ ]:
def chat(prompt):
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("Enter a non-empty prompt.")
    try:
        response = client.invoke_agent_runtime(
            agentRuntimeArn=RUNTIME_ARN,
            qualifier=ENDPOINT,
            runtimeSessionId=session_id,
            contentType="application/json",
            accept="application/json",
            payload=json.dumps({"prompt": prompt}).encode("utf-8"),
        )
    except ClientError as exc:
        error = exc.response.get("Error", {})
        print("AWS error:", error.get("Code"), error.get("Message"))
        raise

    body = response["response"]
    try:
        result = json.loads(body.read())
    finally:
        body.close()

    print("HTTP status:", response.get("statusCode"))
    print("Answer:", result.get("answer", result))
    print("Saved draft:")
    print(json.dumps(result.get("order"), indent=2, ensure_ascii=False))
    return result


## First turn: save a coffee draft

Check the actual returned order and tool calls. One cold coffee should total 12000 paise (₹120). The model response itself is not proof that a write succeeded.


In [ ]:
first = chat("Hi.")


## Second turn: preserve the coffee and add a sandwich

This cell uses the same session. One cold coffee plus one veg sandwich should total 22000 paise (₹220).


In [ ]:
second = chat("Keep my coffee and add one veg sandwich to my draft.")


## Inspect tool evidence


In [ ]:
print(json.dumps(second.get("steps", []), indent=2, ensure_ascii=False))


## Your next message

Edit the prompt and run this cell again to continue the same conversation.


In [ ]:
result = chat("What is in my current draft order?")


## Troubleshooting

- **ProfileNotFound / NoCredentialsError:** configure the `kiosk` profile in a terminal with `aws configure --profile kiosk`.
- **AccessDeniedException:** verify that `InvokeKioskRuntime` permits `bedrock-agentcore:InvokeAgentRuntime` on this DEFAULT endpoint. Also check the account, region, and inbound IAM authentication setting.
- **Model/provider error:** verify `OPEN_ROUTER_API_KEY` and credit in the deployed runtime configuration. The notebook does not read your local `.env`.
- **Missing draft/history:** check session reuse; temporary runtime storage can be lost when the session is replaced or ends.
- **Import errors after installation:** restart the notebook kernel and run the connection cells again.

The standard invocation calls `/invocations`; this notebook does not call the auxiliary local REST routes `/menu`, `/order`, or `/reset`.

Reference: [AWS InvokeAgentRuntime API](https://docs.aws.amazon.com/botocore/latest/reference/services/bedrock-agentcore/client/invoke_agent_runtime.html).


In [ ]:
print(aws_session.client("sts").get_caller_identity())
print(RUNTIME_ARN)
print(ENDPOINT)